# 01 Corpus Normalisation & Metadata Audit
**Project:** `energy-audit`

---
### Purpose
Validate the entire corpus **before** ingestion.

---
## 1 - Imports & paths

In [1]:
import os, re, json, hashlib, logging, sys
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any

import fitz          # PyMuPDF
import yaml
from dotenv import load_dotenv
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.progress import track
from rich import print as rprint

load_dotenv()
console = Console()
logging.basicConfig(level=logging.WARNING)

def _repo_root() -> "Path":
    env = os.getenv("ENERGY_AUDIT_ROOT")
    if env and Path(env).exists():
        return Path(env).resolve()
    cur = Path.cwd().resolve()
    for cand in [cur, *cur.parents]:
        if (cand / "notebooks").is_dir() and (cand / "data").is_dir():
            return cand
    return cur.parent

REPO_ROOT    = _repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
import common as c  # noqa: E402  (shared corpus/llm helpers)
OUTPUTS     = REPO_ROOT / "outputs"
META_PATH   = OUTPUTS   / "corpus_metadata.json"

# ── Layer roots
RAW_EU      = REPO_ROOT / "data" / "raw"       / "eu"

# ── Thresholds
MIN_PDF_BYTES  = 50_000
MIN_PDF_PAGES  = 2
MIN_MD_CHARS   = 500


# ── Required metadata fields per document
REQUIRED_META_FIELDS = [
    "filename", "folder_key", "instrument_type", "domain",
    "short_tag", "jurisdiction", "full_title",
    "effective_date", "in_force_date", "applicability", "status",
    "compliance_risk_level", "key_articles_for_audit", "tags",
]

OUTPUTS.mkdir(parents=True, exist_ok=True)

rprint("[bold]Repo root :[/bold] <repo root (see .env / cwd)>")
rprint(f"[bold]Metadata  :[/bold] {META_PATH.relative_to(REPO_ROOT)}")

Repo root : <repo root (see .env / cwd)>

Metadata  : outputs/corpus_metadata.json

---
## 2 - Load corpus metadata
The canonical source   `outputs/corpus_metadata.json`   defines every expected
document, its folder, filename, and all metadata fields.

In [2]:
if not META_PATH.exists():
    console.print(
        f"[bold red]✗ corpus_metadata.json not found at {META_PATH}[/bold red]\n"
        "  Place the file in outputs/ before running this notebook."
    )
    raise FileNotFoundError(META_PATH)

with META_PATH.open() as f:
    CORPUS_META = json.load(f)

DOCUMENTS: list[dict] = CORPUS_META["documents"]
FOLDER_MAP: dict[str, str] = CORPUS_META["folder_map"]

# Build lookup helpers
BY_TAG  = {d["short_tag"]: d for d in DOCUMENTS}
BY_FILE = {d["filename"]:  d for d in DOCUMENTS}

console.print("[bold green]✓ Loaded corpus_metadata.json[/bold green]")
console.print(f"  Schema version  : {CORPUS_META.get('_schema_version', '?')}")
console.print(f"  Total documents : {len(DOCUMENTS)}")
console.print(f"  Last updated    : {CORPUS_META.get('_last_updated', '?')}")

✓ Loaded corpus_metadata.json

Schema version  : 1.1

Total documents : 25

Last updated    : 2026-08-27T15:17:35

---
## 3   Audit helpers

In [3]:
@dataclass
class DocumentAudit:
    """Collects all audit findings for a single document."""
    doc:             dict
    raw_path:        Path

    # presence
    raw_present:     bool  = False

    # integrity
    size_bytes:      int   = 0
    page_count:      int   = 0
    sha256:          str   = ""
    pdf_valid:       bool  = False

    # PDF quality for RAG
    multi_column:    bool  = False
    has_footnotes:   bool  = False
    has_tables:      bool  = False
    has_headers:     bool  = False
    hyphenation:     bool  = False

    # metadata completeness
    missing_fields:  list  = field(default_factory=list)
    empty_fields:    list  = field(default_factory=list)

    # warnings / errors
    warnings:        list  = field(default_factory=list)
    errors:          list  = field(default_factory=list)

    @property
    def metadata_score(self) -> float:
        """0-1 fraction of required metadata fields that are present and non-empty."""
        bad = len(self.missing_fields) + len(self.empty_fields)
        return max(0.0, 1.0 - bad / len(REQUIRED_META_FIELDS))

    @property
    def rag_issue_count(self) -> int:
        return sum([self.multi_column, self.has_footnotes,
                    self.has_tables, self.has_headers, self.hyphenation])

    @property
    def overall_ok(self) -> bool:
        return (
            self.raw_present
            and self.pdf_valid
            and len(self.errors) == 0
            and self.metadata_score >= 0.9
        )


def resolve_paths(doc: dict) -> tuple[Path, Path]:
    """Return (raw_path) for a document entry."""
    folder_key = doc["folder_key"]
    raw_dir    = REPO_ROOT / FOLDER_MAP[folder_key]
    raw_path   = raw_dir   / doc["filename"]
    return raw_path


console.print("[bold green]✓ Audit helpers ready.[/bold green]")

✓ Audit helpers ready.

---
## 4 - Directory structure check

In [4]:
EXPECTED_DIRS: list[Path] = (
    [REPO_ROOT / p for p in FOLDER_MAP.values()]
)

t = Table(title="Directory structure", header_style="bold magenta", show_lines=False)
t.add_column("Path",    no_wrap=True)
t.add_column("Layer",   width=10)
t.add_column("Status",  justify="center", width=8)

dir_errors = []
for d in sorted(set(EXPECTED_DIRS)):
    rel   = d.relative_to(REPO_ROOT)
    layer = "raw"
    if d.exists():
        sym = "[green]✓[/green]"
    else:
        sym = "[red]✗ missing[/red]"
        dir_errors.append(str(rel))
    t.add_row(str(rel), layer, sym)

console.print(t)

if dir_errors:
    console.print(f"[bold yellow]⚠ {len(dir_errors)} missing director(ies). Run:[/bold yellow]")
    for d in dir_errors:
        rprint(f"  mkdir -p {d}")
else:
    console.print("[bold green]✓ All directories present.[/bold green]")

                  Directory structure                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Path                         ┃ Layer      ┃  Status  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━┩
│ data/raw/eu/en/directives    │ raw        │    ✓     │
│ data/raw/eu/en/guidance      │ raw        │    ✓     │
│ data/raw/eu/en/network_codes │ raw        │    ✓     │
│ data/raw/eu/en/regulations   │ raw        │    ✓     │
└──────────────────────────────┴────────────┴──────────┘

✓ All directories present.

---
## 5 - File presence & integrity

In [5]:
def audit_pdf_integrity(a: DocumentAudit) -> None:
    """ Populate size, page_count, sha256, pdf_valid on the audit object"""

    p = a.raw_path
    if not p.exists():
        a.errors.append("PDF not found")
        return
    
    a.raw_present = True
    data = p.read_bytes()
    a.size_bytes = len(data)
    a.sha256 = hashlib.sha256(data).hexdigest()

    if a.size_bytes < MIN_PDF_BYTES:
        a.errors.append(f"File too small ({a.size_bytes // 1024} KB) - possible error page")
        return
    
    try:
        doc = fitz.open(str(p))
        a.page_count = doc.page_count
        a.pdf_valid = True
        doc.close()
    except Exception as exc:
        a.errors.append(f"PyMuPDF failed: {exc}")
        return
    
    if a.page_count < MIN_PDF_PAGES:
        a.warnings.append(f"Only {a.page_count} page(s), too short")


def audit_pdf_rag_quality(a: DocumentAudit, sample_pages: int=3) -> None:
    """ Detect PDF layout features that could degrade RAG quality"""

    if not a.pdf_valid:
        return
    try:
        doc = fitz.open(str(a.raw_path))
    except Exception:
        return
    
    for i in range(min(sample_pages, doc.page_count)):
        page = doc[i]
        blocks = page.get_text("blocks")
        text = page.get_text("text")

        # Two-column detection
        x0s = [b[0] for b in blocks if b[6] == 0]
        if x0s:
            w = page.rect.width
            if (sum(1 for x in x0s if x < w * 0.45) >= 2 and
                    sum(1 for x in x0s if x > w * 0.50) >= 2):
                a.multi_column = True

        if re.search(r"\(\d{1,3}\)|^\d{1,3}\s",  text, re.MULTILINE): a.has_footnotes = True
        if any(b[6] == 1 for b in blocks):                               a.has_tables    = True
        if [b for b in blocks if b[6]==0 and b[1] < page.rect.height*0.08]: a.has_headers = True
        if re.search(r"\w-\n", text):                                    a.hyphenation   = True

    doc.close()


console.print("[bold green]✓ Integrity checkers ready.[/bold green]")


✓ Integrity checkers ready.

---
## 6 - Metadata completeness checker

In [6]:
def audit_metadata_completeness(a: DocumentAudit) -> None:
    """Check all required fields exist in corpus_metadata.json and are non-empty."""
    doc = a.doc
    for field_name in REQUIRED_META_FIELDS:
        if field_name not in doc:
            a.missing_fields.append(field_name)
        else:
            val = doc[field_name]
            # Empty string, empty list, or None
            if val is None or val == "" or val == []:
                a.empty_fields.append(field_name)

    # Warn on missing optional but recommended fields
    recommended = ["celex", "oj_reference", "smart_contract_relevance",
                   "amends", "amended_by"]
    missing_rec = [f for f in recommended if f not in doc or doc[f] is None]
    if missing_rec:
        a.warnings.append(f"Recommended fields missing: {missing_rec}")


console.print("[bold green]✓ Metadata completeness checker ready.[/bold green]")

✓ Metadata completeness checker ready.

---
## 7 - Run full audit on all 19 documents

In [7]:
audits: list[DocumentAudit] = []

for doc in track(DOCUMENTS, description="Auditing corpus..."):
    raw_path = resolve_paths(doc)
    a = DocumentAudit(doc=doc, raw_path=raw_path)

    audit_pdf_integrity(a)
    audit_pdf_rag_quality(a)
    audit_metadata_completeness(a)

    audits.append(a)

n_ok   = sum(1 for a in audits if a.overall_ok)
n_warn = sum(1 for a in audits if a.warnings and a.overall_ok)
n_fail = sum(1 for a in audits if not a.overall_ok)

console.print(f"\n[bold]Audit complete:[/bold] "
              f"[green]{n_ok} pass[/green]  "
              f"[yellow]{n_warn} with warnings[/yellow]  "
              f"[red]{n_fail} fail[/red]")

Output()

Audit complete: 25 pass  25 with warnings  0 fail

---
## 8 - Cross-check: manifest ↔ disk
Flags PDFs on disk missing from the manifest and manifest entries with no file,
then saves the full audit to `outputs/corpus_audit.json` for downstream notebooks.

In [8]:
all_disk_pdfs = sorted(
    p for p in (RAW_EU / "en").rglob("*.pdf") if p.suffix.lower() == ".pdf"
)
if not all_disk_pdfs:  # legacy layout fallback
    all_disk_pdfs = sorted(p for p in RAW_EU.rglob("*.pdf") if p.suffix.lower() == ".pdf")

manifest_files = {d["filename"] for d in DOCUMENTS}
untracked = [p for p in all_disk_pdfs if p.name not in manifest_files]

missing_on_disk = [d["filename"] for d in DOCUMENTS if not resolve_paths(d).exists()]

t2 = Table(title="Manifest ↔ disk cross-check", header_style="bold magenta", show_lines=False)
t2.add_column("Filename", no_wrap=True)
t2.add_column("Issue", width=32)
t2.add_column("Location", no_wrap=True)

for p in untracked:
    t2.add_row(p.name, "[yellow]untracked by manifest[/yellow]",
               str(p.parent.relative_to(REPO_ROOT)))
for f in missing_on_disk:
    t2.add_row(f, "[red]no PDF on disk[/red]", " ")

console.print(t2)
if untracked or missing_on_disk:
    console.print(f"[bold yellow]⚠ {len(untracked)} untracked / {len(missing_on_disk)} missing on disk   "
                  "update outputs/corpus_metadata.json.[/bold yellow]")
else:
    console.print("[bold green]✓ Manifest and disk are in sync.[/bold green]")

AUDIT_OUT = OUTPUTS / "corpus_audit.json"
payload = {
    "generated": datetime.now().isoformat(),
    "repo_root": ".",
    "documents": [
        {
            "filename": a.doc["filename"],
            "raw_present": a.raw_present,
            "pdf_valid": a.pdf_valid,
            "page_count": a.page_count,
            "size_bytes": a.size_bytes,
            "sha256": a.sha256,
            "rag_flags": {
                "multi_column": a.multi_column, "has_footnotes": a.has_footnotes,
                "has_tables": a.has_tables, "has_headers": a.has_headers,
                "hyphenation": a.hyphenation,
            },
            "metadata_score": round(a.metadata_score, 3),
            "missing_fields": a.missing_fields,
            "warnings": a.warnings,
            "errors": a.errors,
            "overall_ok": a.overall_ok,
        } for a in audits
    ],
    "untracked_pdfs": [str(p.relative_to(REPO_ROOT)) for p in untracked],
    "missing_on_disk": missing_on_disk,
}
with AUDIT_OUT.open("w") as f:
    json.dump(payload, f, indent=2)
console.print(f"[bold green]✓ Corpus audit saved to {AUDIT_OUT.relative_to(REPO_ROOT)}[/bold green]")

               Manifest ↔ disk cross-check                
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Filename ┃ Issue                            ┃ Location ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
└──────────┴──────────────────────────────────┴──────────┘

✓ Manifest and disk are in sync.

✓ Corpus audit saved to outputs/corpus_audit.json